In [4]:
from src.prep.data_loader_01 import ClinicalTrialLoader
from nlp_audit import audit_nlp_pillars


# 1. Initialize the loader with the path to your AACT data
# Replace 'data/' with your actual folder path
loader = ClinicalTrialLoader(data_path='data/')

# 2. Run the core pipeline
# This applies the filters (Industry, Phase, etc.)
df_core = loader.load_and_clean()

# 3. Run the feature engineering
# This triggers _prepare_text and creates the pillars
df_features = loader.add_features(df_core)

# 4. RUN THE AUDIT
# This is the code I provided in the previous response
audit_nlp_pillars(loader, df_features)

>>> 1. Loading Studies & Applying Filters...
    [Filter] Kept 125576 Industry-led trials.
    [Sanitizer] Dropping 168 trials terminated due to COVID/Logistics.
    Core Cohort: 33929 trials (Phase 1/2/3, 2005-2025 training window and 2005-2025 for production).
>>> 2. Engineering Features...
    -> Grouping Phases into Efficacy Tiers...
    -> Engineering Sponsor Tiers...
    -> Engineering Protocol Complexity (Calculating Age Flags)...
    -> Attaching Medical Hierarchy (Preserving Tree Structure)...
    -> Engineering NLP Text Pillars (Scientific & Operational)...
    -> Engineering Agent Type (Prioritizing Active Molecules)...
    -> Engineering Smart Competition (Disease & Molecule density)...
    -> Engineering Smart Patterns (Rigor & Strictness)...
    -> Engineering Gated Protocol Features (FDA & DMC)...
    -> Attaching Vectorized Text Embeddings...
       Attached 20 dimensions.
    -> Attaching P-Values (Scientific Success Logic)...
       [Audit] Trials with P-values: 7532 

In [5]:
def audit_text_density(df):
    """
    Analyzes the length and information density of the three NLP pillars.
    """
    pillars = ['txt_scientific_essence', 'txt_criteria', 'txt_primary_endpoints']
    stats = {}

    print("="*80)
    print("NLP PILLAR DENSITY & TRUNCATION AUDIT")
    print("="*80)

    for col in pillars:
        # Calculate word counts (rough proxy for tokens)
        word_counts = df[col].apply(lambda x: len(str(x).split()))

        # BERT uses WordPiece, so 1 word is roughly 1.3 tokens
        est_tokens = word_counts * 1.3
        truncation_rate = (est_tokens > 512).mean()

        stats[col] = {
            'mean_words': word_counts.mean(),
            'max_words': word_counts.max(),
            'median_words': word_counts.median(),
            'truncation_risk': truncation_rate
        }

        print(f"\n>>> Pillar: {col}")
        print(f"    - Average Length: {word_counts.mean():.1f} words")
        print(f"    - Median Length:  {word_counts.median():.1f} words")
        print(f"    - Truncation Risk: {truncation_rate:.1%} of trials exceed 512-token limit")

        # Check for placeholder dominance
        placeholder = f"No {col.replace('txt_', '')} provided"
        placeholder_pct = (df[col] == placeholder).mean()
        print(f"    - Missing Data:   {placeholder_pct:.1%} use placeholders")

    return stats

# Run the audit
pillar_stats = audit_text_density(df_features)

NLP PILLAR DENSITY & TRUNCATION AUDIT

>>> Pillar: txt_scientific_essence
    - Average Length: 91.4 words
    - Median Length:  76.0 words
    - Truncation Risk: 0.3% of trials exceed 512-token limit
    - Missing Data:   0.0% use placeholders

>>> Pillar: txt_criteria
    - Average Length: 343.7 words
    - Median Length:  261.0 words
    - Truncation Risk: 32.8% of trials exceed 512-token limit
    - Missing Data:   0.0% use placeholders

>>> Pillar: txt_primary_endpoints
    - Average Length: 26.3 words
    - Median Length:  14.0 words
    - Truncation Risk: 0.4% of trials exceed 512-token limit
    - Missing Data:   0.4% use placeholders
